In [ ]:
!pip install dash==2.14.2 pandas plotly openpyxl scikit-learn -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error

# ════════════════════════════════════════════════════════
# PALETTE
# ════════════════════════════════════════════════════════
NAVY  = '#1F3864'
STEEL = '#5B7FA6'
SAGE  = '#7A9E7E'
TERRA = '#8B5E52'
AMBER = '#C17F3A'
GREY  = '#888888'
LIGHT = '#F4F6FB'
WHITE = '#FFFFFF'

DIV_COLORS = {
    'West':        '#4A7B9D',
    'Northeast':   '#5A7A5A',
    'Southeast':   '#8B7355',
    'Central':     '#7B6B8A',
    'Mid-Atlantic':'#8B5A5A',
}

# ════════════════════════════════════════════════════════
# DATA LOADING & PREPARATION
# ════════════════════════════════════════════════════════
df_raw = pd.read_excel('data.xlsx', header=[0, 1])
df_raw.columns = [
    '_'.join([str(i) for i in col if str(i) != 'nan'])
    for col in df_raw.columns
]

data = pd.DataFrame()
data['Name']          = df_raw[df_raw.columns[1]]
data['Division']      = df_raw[df_raw.columns[2]].str.strip()
data['State']         = df_raw[df_raw.columns[5]]
data['Income']        = df_raw['Median House Hold Income_2025']
data['Income_Growth'] = df_raw['House Hold Income 5-yr Growth_2025']
data['Home_Value']    = df_raw['Avg. Home Value_2025']
data['Education']     = df_raw['% Educated - Bachelors Plus_2025']
data['Population']    = df_raw['Population_2025']
data['Daytime_Pop']   = df_raw['Daytime Population_2025']
data['Households']    = df_raw['Total Households_2025']
data['HH_Growth']     = df_raw['Household 5 Yr Growth_2025']
data = data.dropna().reset_index(drop=True)

data['Purchasing_Power'] = data['Income'] * data['Population']
data['PP_Billions']      = data['Purchasing_Power'] / 1e9

# ════════════════════════════════════════════════════════
# PCA — PURCHASING POWER INDEX
# ════════════════════════════════════════════════════════
pca_vars   = ['Income','Income_Growth','Home_Value','Education',
               'Population','Daytime_Pop','Households','HH_Growth']
var_labels = ['Median Income','Income Growth','Home Value','Education',
              'Population','Daytime Pop','Total Households','HH Growth']

scaler = StandardScaler()
Xs     = scaler.fit_transform(data[pca_vars].values)
pca    = PCA()
pca.fit(Xs)
scores = pca.transform(Xs)
if np.corrcoef(scores[:, 0], data['Population'].values)[0, 1] < 0:
    scores[:, 0] *= -1
    pca.components_[0] *= -1
data['PPI'] = scores[:, 0]
explained   = pca.explained_variance_ratio_
loadings    = pca.components_

# ════════════════════════════════════════════════════════
# MACHINE LEARNING MODELS
# ════════════════════════════════════════════════════════
feat_cols = ['Income', 'Income_Growth', 'Education', 'Population', 'HH_Growth']
X = data[feat_cols]
y = data['PPI']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)
gb = GradientBoostingRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)

lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)
gb_pred = gb.predict(X_test)

lr_r2 = r2_score(y_test, lr_pred);  lr_mae = mean_absolute_error(y_test, lr_pred)
rf_r2 = r2_score(y_test, rf_pred);  rf_mae = mean_absolute_error(y_test, rf_pred)
gb_r2 = r2_score(y_test, gb_pred);  gb_mae = mean_absolute_error(y_test, gb_pred)

cv_lr = cross_val_score(lr, X, y, cv=5, scoring='r2').mean()
cv_rf = cross_val_score(rf, X, y, cv=5, scoring='r2').mean()
cv_gb = cross_val_score(gb, X, y, cv=5, scoring='r2').mean()

importance_df = pd.DataFrame({
    'Feature': ['Population','Income Growth','Education','Median Income','HH Growth'],
    'RF_Imp':  [rf.feature_importances_[feat_cols.index(f)]*100
                for f in ['Population','Income_Growth','Education','Income','HH_Growth']],
    'GB_Imp':  [gb.feature_importances_[feat_cols.index(f)]*100
                for f in ['Population','Income_Growth','Education','Income','HH_Growth']],
}).sort_values('RF_Imp').reset_index(drop=True)

corr_cols = ['Income','Income_Growth','Home_Value','Education',
             'Population','Daytime_Pop','Households','HH_Growth','PPI']
corr = data[corr_cols].corr().round(2)

loadings_df = pd.DataFrame({
    'Variable': var_labels,
    'Loading':  loadings[0]
}).sort_values('Loading').reset_index(drop=True)

print(f"Dataset: {len(data)} trade areas across {data['Division'].nunique()} divisions")
print(f"LR  R²={lr_r2:.3f}  MAE={lr_mae:.4f}  CV={cv_lr:.3f}")
print(f"RF  R²={rf_r2:.3f}  MAE={rf_mae:.4f}  CV={cv_rf:.3f}")
print(f"GB  R²={gb_r2:.3f}  MAE={gb_mae:.4f}  CV={cv_gb:.3f}")

# ════════════════════════════════════════════════════════
# STATIC FIGURES (not affected by filters)
# ════════════════════════════════════════════════════════

# ── Correlation heatmap ───────────────────────────────
fig_corr = px.imshow(
    corr, text_auto=True,
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Figure 5: Correlation Matrix — All Demographic Variables vs. PPI'
)
fig_corr.update_layout(
    font_family='Georgia', title_font_size=13,
    margin=dict(t=55, b=10, l=10, r=10)
)

# ── PCA scree plot ────────────────────────────────────
fig_scree = go.Figure()
fig_scree.add_trace(go.Bar(
    x=[f'PC{i+1}' for i in range(8)],
    y=np.round(explained*100, 1),
    name='Variance %',
    marker_color=[NAVY]+[STEEL]*7,
    text=[f'{v:.1f}%' for v in np.round(explained*100, 1)],
    textposition='outside'
))
fig_scree.add_trace(go.Scatter(
    x=[f'PC{i+1}' for i in range(8)],
    y=np.round(np.cumsum(explained)*100, 1),
    name='Cumulative %', mode='lines+markers',
    line=dict(color=TERRA, width=2), marker=dict(size=6),
    yaxis='y2'
))
fig_scree.update_layout(
    title='Figure 6: PCA Scree Plot — Variance Explained by Component',
    yaxis=dict(title='Variance Explained (%)', range=[0, 45]),
    yaxis2=dict(title='Cumulative (%)', overlaying='y', side='right',
                range=[0, 115], showgrid=False),
    legend=dict(x=0.55, y=0.95, bgcolor='rgba(255,255,255,0.8)'),
    font_family='Georgia', title_font_size=13,
    margin=dict(t=55, b=40, l=10, r=60)
)

# ── PC1 loadings ──────────────────────────────────────
fig_loadings = go.Figure(go.Bar(
    x=loadings_df['Loading'], y=loadings_df['Variable'],
    orientation='h',
    marker_color=[NAVY if v >= 0 else TERRA for v in loadings_df['Loading']],
    text=[f'{v:+.3f}' for v in loadings_df['Loading']],
    textposition='outside'
))
fig_loadings.add_vline(x=0, line_width=1, line_color='#333333')
fig_loadings.update_layout(
    title='Figure 7: PC1 Loadings — What Drives the Purchasing Power Index',
    xaxis=dict(title='Loading on PC1', range=[-0.15, 0.72]),
    font_family='Georgia', title_font_size=13,
    margin=dict(t=55, b=10, l=10, r=70)
)

# ── Feature importance ────────────────────────────────
fig_imp = go.Figure()
fig_imp.add_trace(go.Bar(
    y=importance_df['Feature'], x=importance_df['RF_Imp'],
    name='Random Forest', orientation='h',
    marker_color=NAVY,
    text=[f'{v:.1f}%' for v in importance_df['RF_Imp']],
    textposition='outside'
))
fig_imp.add_trace(go.Bar(
    y=importance_df['Feature'], x=importance_df['GB_Imp'],
    name='Gradient Boosting', orientation='h',
    marker_color=SAGE,
    text=[f'{v:.1f}%' for v in importance_df['GB_Imp']],
    textposition='outside'
))
fig_imp.update_layout(
    title='Figure 9: Feature Importance — Random Forest vs. Gradient Boosting',
    barmode='group',
    xaxis=dict(title='Importance (%)', range=[0, 115]),
    legend=dict(x=0.55, y=0.15, bgcolor='rgba(255,255,255,0.8)'),
    font_family='Georgia', title_font_size=13,
    margin=dict(t=55, b=10, l=10, r=60)
)

# ── Model comparison ──────────────────────────────────
models_df = pd.DataFrame({
    'Model': ['Linear\nRegression', 'Random\nForest', 'Gradient\nBoosting'],
    'R2':    [round(lr_r2,4), round(rf_r2,4), round(gb_r2,4)],
    'MAE':   [round(lr_mae,4), round(rf_mae,4), round(gb_mae,4)],
    'CV':    [round(cv_lr,4), round(cv_rf,4), round(cv_gb,4)],
})
fig_models = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Test R² (higher = better)',
                    'Test MAE, PPI units (lower = better)',
                    '5-Fold CV R² (higher = better)']
)
mc = [STEEL, NAVY, SAGE]
fig_models.add_trace(go.Bar(x=models_df['Model'], y=models_df['R2'],
    marker_color=mc, showlegend=False,
    text=[str(v) for v in models_df['R2']], textposition='outside'), row=1, col=1)
fig_models.add_trace(go.Bar(x=models_df['Model'], y=models_df['MAE'],
    marker_color=mc, showlegend=False,
    text=[str(v) for v in models_df['MAE']], textposition='outside'), row=1, col=2)
fig_models.add_trace(go.Bar(x=models_df['Model'], y=models_df['CV'],
    marker_color=mc, showlegend=False,
    text=[str(v) for v in models_df['CV']], textposition='outside'), row=1, col=3)
fig_models.update_yaxes(range=[0.84, 0.98], row=1, col=1)
fig_models.update_yaxes(range=[0,    0.26],  row=1, col=2)
fig_models.update_yaxes(range=[0.84, 0.98], row=1, col=3)
fig_models.update_layout(
    title_text='Figure 8: Model Performance Comparison — Test Set + Cross-Validation',
    font_family='Georgia', title_font_size=13,
    margin=dict(t=70, b=10, l=10, r=10)
)

# ── Actual vs Predicted ───────────────────────────────
def make_avp(y_true, y_pred, title, color):
    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    lo  = min(float(y_true.min()), float(y_pred.min())) - 0.2
    hi  = max(float(y_true.max()), float(y_pred.max())) + 0.2
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(y_true), y=list(y_pred), mode='markers',
        marker=dict(color=color, size=6, opacity=0.65), name='Predictions'
    ))
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[lo, hi], mode='lines',
        line=dict(color='#444444', dash='dash', width=1.5), name='Perfect fit'
    ))
    fig.update_layout(
        title=f'{title}<br><sup>R² = {r2:.3f}   MAE = {mae:.4f}</sup>',
        xaxis_title='Actual PPI', yaxis_title='Predicted PPI',
        legend=dict(x=0.05, y=0.95, bgcolor='rgba(255,255,255,0.8)'),
        font_family='Georgia', title_font_size=13,
        margin=dict(t=65, b=10, l=10, r=10)
    )
    return fig

fig_avp_lr = make_avp(y_test, lr_pred, 'Linear Regression — Actual vs. Predicted', STEEL)
fig_avp_rf = make_avp(y_test, rf_pred, 'Random Forest — Actual vs. Predicted',      NAVY)
fig_avp_gb = make_avp(y_test, gb_pred, 'Gradient Boosting — Actual vs. Predicted',  SAGE)

# ════════════════════════════════════════════════════════
# SHARED STYLES
# ════════════════════════════════════════════════════════
CARD = {
    'background': '#f0f4fa', 'border': '1px solid #d0d8e8',
    'borderRadius': '8px', 'padding': '12px 16px',
    'margin': '4px', 'textAlign': 'center',
    'minWidth': '0', 'flex': '1'
}
VAL  = {'fontSize': '21px', 'fontWeight': 'bold', 'color': NAVY, 'margin': '4px 0 0 0', 'fontFamily': 'Georgia'}
LBL  = {'fontSize': '10px', 'color': GREY, 'textTransform': 'uppercase',
        'letterSpacing': '0.07em', 'margin': '0', 'fontFamily': 'Arial'}
ROW  = {'display': 'flex', 'flexWrap': 'wrap'}
HALF = {'flex': '1', 'minWidth': '420px'}

# ── Story panel style ─────────────────────────────────
def story_panel(title, body_text, accent=NAVY, icon='📋'):
    return html.Div([
        html.Div([
            html.Span(icon, style={'fontSize': '22px', 'marginRight': '10px'}),
            html.Span(title, style={
                'fontSize': '15px', 'fontWeight': 'bold',
                'color': WHITE, 'fontFamily': 'Georgia'
            }),
        ], style={
            'background': accent, 'padding': '10px 16px',
            'borderRadius': '6px 6px 0 0'
        }),
        html.Div(body_text, style={
            'padding': '14px 18px',
            'background': '#fafcff',
            'border': f'1px solid {accent}',
            'borderTop': 'none',
            'borderRadius': '0 0 6px 6px',
            'lineHeight': '1.7',
            'fontSize': '13px',
            'fontFamily': 'Georgia',
            'color': '#333333'
        })
    ], style={'marginBottom': '12px'})

def insight_box(text, color=NAVY):
    return html.Div([
        html.Span('💡 ', style={'fontSize': '15px'}),
        html.Span(text, style={
            'fontSize': '13px', 'fontWeight': 'bold',
            'color': color, 'fontFamily': 'Georgia'
        })
    ], style={
        'background': '#EBF3FB', 'border': f'2px solid {color}',
        'borderRadius': '6px', 'padding': '10px 16px',
        'marginBottom': '10px'
    })

def section_header(num, title, color=NAVY):
    return html.Div([
        html.Span(f'Section {num}', style={
            'fontSize': '11px', 'fontWeight': 'bold', 'color': WHITE,
            'background': color, 'padding': '3px 10px',
            'borderRadius': '12px', 'marginRight': '10px',
            'fontFamily': 'Arial', 'letterSpacing': '0.05em'
        }),
        html.Span(title, style={
            'fontSize': '16px', 'fontWeight': 'bold',
            'color': color, 'fontFamily': 'Georgia'
        }),
    ], style={
        'padding': '14px 20px 10px 20px',
        'borderBottom': f'2px solid {color}',
        'marginBottom': '14px'
    })

SECTION_STYLE = {
    'background': WHITE,
    'borderRadius': '8px',
    'border': '1px solid #dde3ee',
    'margin': '12px 8px',
    'boxShadow': '0 1px 4px rgba(0,0,0,0.07)',
    'overflow': 'hidden',
    'padding': '0 0 14px 0'
}

# ════════════════════════════════════════════════════════
# DASH APP
# ════════════════════════════════════════════════════════
app = Dash(__name__)

app.layout = html.Div([

    # ════════════════════════════════════════════════════
    # HEADER BANNER
    # ════════════════════════════════════════════════════
    html.Div([
        html.Div([
            html.H1(
                'Regency Centers — Trade Area Purchasing Power',
                style={'color': WHITE, 'margin': '0', 'fontSize': '22px', 'fontFamily': 'Georgia'}
            ),
            html.P(
                'An Interactive Predictive Analytics Dashboard  |  CEN6940.14925  |  Group 7  |  Spring 2026',
                style={'color': '#b8cce4', 'margin': '4px 0 0 0', 'fontSize': '12px', 'fontFamily': 'Arial'}
            ),
        ]),
        html.Div([
            html.Span('445', style={'fontSize':'28px','fontWeight':'bold','color':WHITE,'fontFamily':'Georgia'}),
            html.Span(' trade areas  ', style={'fontSize':'13px','color':'#b8cce4','fontFamily':'Arial'}),
            html.Span('|', style={'color':'#5B7FA6','margin':'0 8px'}),
            html.Span('5', style={'fontSize':'28px','fontWeight':'bold','color':WHITE,'fontFamily':'Georgia'}),
            html.Span(' divisions  ', style={'fontSize':'13px','color':'#b8cce4','fontFamily':'Arial'}),
            html.Span('|', style={'color':'#5B7FA6','margin':'0 8px'}),
            html.Span('3', style={'fontSize':'28px','fontWeight':'bold','color':WHITE,'fontFamily':'Georgia'}),
            html.Span(' ML models', style={'fontSize':'13px','color':'#b8cce4','fontFamily':'Arial'}),
        ], style={'textAlign':'right'})
    ], style={
        'background': f'linear-gradient(135deg, {NAVY} 0%, #2E5E8E 100%)',
        'padding': '20px 28px',
        'display': 'flex',
        'justifyContent': 'space-between',
        'alignItems': 'center',
        'marginBottom': '0'
    }),

    # ════════════════════════════════════════════════════
    # PAGE 0 — PROJECT BACKGROUND & CONTEXT
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('0', 'Project Background & Context', NAVY),
        html.Div([

            # Left: background
            html.Div([
                story_panel(
                    'About This Project', [
                        html.P([
                            html.B('Regency Centers '), 'is a leading U.S. owner, operator, and developer of '
                            'grocery-anchored shopping centers. The company currently evaluates trade areas using '
                            'its proprietary "DNA" model — a descriptive framework based on household income, '
                            'population, and growth rates. While informative, this model is static and lacks '
                            'predictive capability.'
                        ]),
                        html.P(
                            'This project replaces descriptive analysis with a full predictive analytics pipeline: '
                            'Exploratory Data Analysis → PCA-based Purchasing Power Index construction → '
                            'Machine Learning modelling → Interactive data storytelling.'
                        ),
                        html.P([
                            html.B('Analytical Framework (Rogers et al., 2004): '),
                            html.Span('Purchasing Power = Income Capacity × Market Size', style={'fontStyle':'italic','color':NAVY})
                        ]),
                    ], icon='🏢', accent=NAVY
                ),
                story_panel(
                    'Dataset Overview', [
                        html.Ul([
                            html.Li('530 original trade area records from the Regency Centers demographic dataset'),
                            html.Li([html.B('445 valid trade areas '), 'retained after cleaning (removing missing values)']),
                            html.Li('8 demographic & economic variables (2025 projections)'),
                            html.Li('5 geographic divisions: West (90), Northeast (95), Southeast (136), Central (74), Mid-Atlantic (50)'),
                            html.Li('Historical data available from 2017 through 2025'),
                        ], style={'paddingLeft':'20px', 'margin':'0'})
                    ], icon='📊', accent=STEEL
                ),
            ], style={'flex':'1', 'minWidth':'360px', 'padding':'0 8px'}),

            # Right: research questions + variables
            html.Div([
                story_panel(
                    'Four Research Questions', [
                        html.Ol([
                            html.Li('Which demographic and economic variables most predictively drive trade area purchasing power?'),
                            html.Li('What is a consistent, scalable approach to measuring purchasing power across the portfolio?'),
                            html.Li('What growth and decline trends characterize purchasing power across trade areas?'),
                            html.Li('How accurately and reliably can purchasing power be predicted from demographic variables?'),
                        ], style={'paddingLeft':'20px', 'margin':'0'})
                    ], icon='❓', accent=TERRA
                ),
                story_panel(
                    'Eight Variables Used in Analysis', [
                        html.Div([
                            html.Div([
                                html.P('INCOME CAPACITY', style={'fontSize':'10px','fontWeight':'bold','color':GREY,'margin':'0 0 6px 0','letterSpacing':'0.07em'}),
                                html.Ul([
                                    html.Li('Median Household Income (2025)'),
                                    html.Li('5-Year Income Growth Projection'),
                                    html.Li('Average Home Value (2025)'),
                                    html.Li('Education Attainment (% Bachelor\'s+)'),
                                ], style={'paddingLeft':'16px','margin':'0','fontSize':'12px'})
                            ], style={'flex':'1'}),
                            html.Div([
                                html.P('MARKET SIZE', style={'fontSize':'10px','fontWeight':'bold','color':GREY,'margin':'0 0 6px 0','letterSpacing':'0.07em'}),
                                html.Ul([
                                    html.Li('Residential Population (2025)'),
                                    html.Li('Daytime Population (2025)'),
                                    html.Li('Total Households (2025)'),
                                    html.Li('5-Year Household Growth'),
                                ], style={'paddingLeft':'16px','margin':'0','fontSize':'12px'})
                            ], style={'flex':'1'}),
                        ], style={'display':'flex','gap':'16px'})
                    ], icon='📋', accent=SAGE
                ),
            ], style={'flex':'1', 'minWidth':'360px', 'padding':'0 8px'}),

        ], style={**ROW, 'padding':'0 8px'}),
    ], style=SECTION_STYLE),

    # ════════════════════════════════════════════════════
    # FILTERS + KPI
    # ════════════════════════════════════════════════════
    html.Div([
        html.Div([
            html.Label('Filter by Division',
                style={'fontWeight':'bold','fontSize':'12px','color':NAVY,
                       'display':'block','marginBottom':'4px','fontFamily':'Arial'}),
            dcc.Dropdown(
                id='div-filter',
                options=([{'label':'All Divisions','value':'ALL'}]+
                         [{'label':d,'value':d} for d in sorted(data['Division'].unique())]),
                value='ALL', clearable=False,
                style={'fontSize':'13px','width':'220px'}
            )
        ], style={'marginRight':'32px'}),
        html.Div([
            html.Label(
                f'Filter by Income Range  (${int(data["Income"].min()/1000)}k – ${int(data["Income"].max()/1000)}k)',
                style={'fontWeight':'bold','fontSize':'12px','color':NAVY,
                       'display':'block','marginBottom':'4px','fontFamily':'Arial'}
            ),
            dcc.RangeSlider(
                id='income-slider',
                min=int(data['Income'].min()), max=int(data['Income'].max()),
                value=[int(data['Income'].min()), int(data['Income'].max())],
                marks={
                    int(data['Income'].quantile(0)):    f"${int(data['Income'].quantile(0)/1000)}k",
                    int(data['Income'].quantile(0.25)): f"${int(data['Income'].quantile(0.25)/1000)}k",
                    int(data['Income'].quantile(0.5)):  f"${int(data['Income'].quantile(0.5)/1000)}k",
                    int(data['Income'].quantile(0.75)): f"${int(data['Income'].quantile(0.75)/1000)}k",
                    int(data['Income'].quantile(1)):    f"${int(data['Income'].quantile(1)/1000)}k",
                },
                tooltip={'placement':'bottom','always_visible':False}
            )
        ], style={'flex':'1'}),
    ], style={
        'display':'flex','alignItems':'flex-start',
        'padding':'14px 18px','background':'#EDF2FA',
        'borderRadius':'6px','margin':'8px 8px 4px 8px',
        'border':f'1px solid #c8d4e8'
    }),

    # KPI cards
    html.Div([
        html.Div([html.P('Trade Areas',    style=LBL), html.P(id='kpi-count',   style=VAL)], style=CARD),
        html.Div([html.P('Avg Income',     style=LBL), html.P(id='kpi-income',  style=VAL)], style=CARD),
        html.Div([html.P('Avg Population', style=LBL), html.P(id='kpi-pop',     style=VAL)], style=CARD),
        html.Div([html.P('Avg PPI Score',  style=LBL), html.P(id='kpi-ppi',     style=VAL)], style=CARD),
    ], style={'display':'flex','margin':'4px 4px 0 4px'}),
    html.Div([
        html.Div([html.P('Total PP ($B)',     style=LBL), html.P(id='kpi-pp',      style=VAL)], style=CARD),
        html.Div([html.P('Avg Income Growth', style=LBL), html.P(id='kpi-incgrow', style=VAL)], style=CARD),
        html.Div([html.P('Avg HH Growth',     style=LBL), html.P(id='kpi-hhgrow',  style=VAL)], style=CARD),
    ], style={'display':'flex','margin':'4px 4px 12px 4px'}),

    # ════════════════════════════════════════════════════
    # SECTION 1 — MARKET OVERVIEW
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('1', 'Market Overview — Portfolio Purchasing Power', NAVY),
        html.Div([
            insight_box(
                'Key Finding: West and Mid-Atlantic divisions deliver the highest average purchasing power per '
                'trade area ($18.3B and $18.9B). Southeast has the most properties (136) but the lowest '
                'average ($8.5B) — reflecting broad suburban coverage rather than density concentration.',
                NAVY
            ),
        ], style={'padding':'0 16px'}),
        html.Div([
            html.Div([dcc.Graph(id='bar-top-markets')], style=HALF),
            html.Div([dcc.Graph(id='bar-div-pp')],      style=HALF),
        ], style=ROW),
        html.Div([
            html.Div([dcc.Graph(id='bar-div-ppi')],  style=HALF),
            html.Div([dcc.Graph(id='box-div-ppi')],  style=HALF),
        ], style=ROW),
        html.Div([
            story_panel(
                'What These Charts Show', [
                    html.P([html.B('Top 10 Markets (upper left): '),
                        'The two New York City properties — 1225–1239 Second Ave ($167.5B) and 1175 Third Avenue '
                        '($165.2B) — dwarf all other markets. Their combined purchasing power ($332.7B) exceeds '
                        'that of the entire Southeast division ($1.16T across 136 properties on a per-property basis). '
                        'This extreme concentration requires a distinct investment and tenant strategy.']),
                    html.P([html.B('Division Box Plot (lower right): '),
                        'Northeast shows the widest distribution with the highest outliers (PPI = 18), while West '
                        'and Mid-Atlantic show tight distributions consistently above the portfolio mean — '
                        'reliable, not outlier-driven performance. Southeast is narrow and below the mean.'])
                ], icon='📈', accent=STEEL
            )
        ], style={'padding':'0 16px'}),
    ], style=SECTION_STYLE),

    # ════════════════════════════════════════════════════
    # SECTION 2 — DEMOGRAPHIC DRIVERS
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('2', 'Demographic Drivers — What Predicts Purchasing Power?', STEEL),
        html.Div([
            insight_box(
                'Critical Finding: Population (r = 0.937) and Daytime Population (r = 0.928) are '
                'overwhelmingly stronger predictors of purchasing power than Median Income (r = 0.108). '
                'Market volume — not income per household — drives aggregate retail spending capacity.',
                STEEL
            ),
        ], style={'padding':'0 16px'}),
        html.Div([
            html.Div([dcc.Graph(id='scatter-pop')],    style=HALF),
            html.Div([dcc.Graph(id='scatter-income')], style=HALF),
        ], style=ROW),
        html.Div([
            html.Div([dcc.Graph(id='scatter-daytime')], style=HALF),
            html.Div([dcc.Graph(id='scatter-growth')],  style=HALF),
        ], style=ROW),
        html.Div([
            story_panel(
                'How to Read These Charts', [
                    html.P([html.B('Population vs. PP (upper left): '),
                        'A steep, tight relationship — the larger the residential market, '
                        'the greater the aggregate spending capacity. West and Northeast trade areas '
                        'dominate the upper right; Southeast clusters in the lower left.']),
                    html.P([html.B('Income vs. PP (upper right): '),
                        'The relationship is nearly flat. High-income trade areas scatter widely across all '
                        'purchasing power levels. A $267k-income area with 20,000 residents generates far less '
                        'total spending than a $90k-income area with 500,000 residents.']),
                    html.P([html.B('Growth Bubble (lower right): '),
                        'Markets in the upper-right quadrant — above-average income AND household growth '
                        'simultaneously — are emerging strategic opportunities. Small bubbles in this quadrant '
                        'have modest current purchasing power but strong dual-growth momentum, making them '
                        'ideal for early investment ahead of demand growth.']),
                ], icon='🔍', accent=STEEL
            ),
            dcc.Graph(figure=fig_corr, style={'width':'100%'}),
        ], style={'padding':'0 16px'}),
    ], style=SECTION_STYLE),

    # ════════════════════════════════════════════════════
    # SECTION 3 — PCA & PURCHASING POWER INDEX
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('3', 'Principal Component Analysis — The Purchasing Power Index (PPI)', TERRA),
        html.Div([
            insight_box(
                'PCA Finding: PC1 (the PPI) explains 37.6% of total portfolio variance and loads primarily '
                'on market size variables. The near-zero loading of Median Income (−0.027) confirms that '
                'income per household and market volume are nearly orthogonal — independent strategic dimensions.',
                TERRA
            ),
        ], style={'padding':'0 16px'}),
        html.Div([
            story_panel(
                'Why PCA? Why the PPI?', [
                    html.P(
                        'The conventional Income × Population measure combines only two variables. PCA '
                        'constructs the Purchasing Power Index (PPI) by simultaneously weighting all eight '
                        'demographic variables according to their natural covariance structure — producing a '
                        'composite index that is data-driven rather than assumed.'
                    ),
                    html.P([
                        'The PPI correlates with Income × Population at r = 0.96 (confirming both capture '
                        'the same phenomenon), but is methodologically superior because it: ',
                        html.Br(),
                        '① Incorporates all eight variables simultaneously  ',
                        html.Br(),
                        '② Resolves multicollinearity by construction through orthogonal components  ',
                        html.Br(),
                        '③ Reveals the two-dimensional portfolio structure (PC1 = market size; PC2 = income quality)'
                    ]),
                ], icon='🧮', accent=TERRA
            ),
        ], style={'padding':'0 16px'}),
        html.Div([
            html.Div([dcc.Graph(figure=fig_scree)],    style=HALF),
            html.Div([dcc.Graph(figure=fig_loadings)], style=HALF),
        ], style=ROW),
        html.Div([
            html.Div([dcc.Graph(id='scatter-ppi')], style=HALF),
            html.Div([dcc.Graph(id='hist-ppi')],    style=HALF),
        ], style=ROW),
        html.Div([
            story_panel(
                'The Two-Dimensional Portfolio Strategy Framework', [
                    html.P(
                        'Because PC1 (market size) and PC2 (income quality) are orthogonal, every trade area '
                        'can be positioned in a two-dimensional space. This reveals four strategic quadrants:'
                    ),
                    html.Div([
                        html.Div([
                            html.B('① Premium Urban Core', style={'color':NAVY}),
                            html.Br(), html.Span('High PC1 + High PC2', style={'fontSize':'11px','color':GREY}),
                            html.P('Flagship tenants, premium mix, highest investment priority', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#EBF3FB','borderRadius':'6px','margin':'4px'}),
                        html.Div([
                            html.B('② Volume Market', style={'color':STEEL}),
                            html.Br(), html.Span('High PC1 + Low PC2', style={'fontSize':'11px','color':GREY}),
                            html.P('Grocery anchors, value retail, high-traffic convenience', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#EBF3FB','borderRadius':'6px','margin':'4px'}),
                        html.Div([
                            html.B('③ Affluent Suburban', style={'color':TERRA}),
                            html.Br(), html.Span('Low PC1 + High PC2', style={'fontSize':'11px','color':GREY}),
                            html.P('Specialty retail, experiential, boutique service', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#FDF0EB','borderRadius':'6px','margin':'4px'}),
                        html.Div([
                            html.B('④ Standard Suburban', style={'color':GREY}),
                            html.Br(), html.Span('Low PC1 + Low PC2', style={'fontSize':'11px','color':GREY}),
                            html.P('Core grocery anchor, operational efficiency focus', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#F5F5F5','borderRadius':'6px','margin':'4px'}),
                    ], style={'display':'flex','flexWrap':'wrap'})
                ], icon='🗺️', accent=TERRA
            )
        ], style={'padding':'0 16px'}),
    ], style=SECTION_STYLE),

    # ════════════════════════════════════════════════════
    # SECTION 4 — PREDICTIVE MODELS
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('4', 'Predictive Models — Machine Learning Results', SAGE),
        html.Div([
            insight_box(
                'Model Result: All three models achieve test R² above 0.91 on genuinely held-out data. '
                'Gradient Boosting (R² = 0.936, MAE = 0.163 PPI units) is recommended for operational '
                'trade area scoring. Linear Regression (CV R² = 0.947) is recommended for stakeholder '
                'communication due to its interpretable coefficients.',
                SAGE
            ),
        ], style={'padding':'0 16px'}),
        html.Div([
            story_panel(
                'Why Three Models?', [
                    html.Div([
                        html.Div([
                            html.B('Linear Regression', style={'color':STEEL}),
                            html.P('Provides interpretable coefficients that explain why a specific trade area '
                                   'scores as it does. Best cross-validation R² (0.947) confirms stable '
                                   'generalisation. Essential for stakeholder communication.', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#EBF3FB','borderRadius':'6px','margin':'4px'}),
                        html.Div([
                            html.B('Random Forest', style={'color':NAVY}),
                            html.P('200 decision trees on bootstrapped samples. Captures non-linear interactions '
                                   'without parametric assumptions. Directly addresses the normality and '
                                   'homoscedasticity violations found in regression assumption testing.', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#EBF3FB','borderRadius':'6px','margin':'4px'}),
                        html.Div([
                            html.B('Gradient Boosting ✓ Recommended', style={'color':SAGE}),
                            html.P('200 sequential trees — each corrects the residual errors of the previous '
                                   'ensemble. Achieves the lowest test MAE (0.163 PPI units) and highest test R². '
                                   'Recommended for operational trade area scoring.', style={'fontSize':'12px','margin':'4px 0 0 0'}),
                        ], style={'flex':'1','padding':'8px','background':'#EBF7EF','borderRadius':'6px','margin':'4px'}),
                    ], style={'display':'flex','flexWrap':'wrap'})
                ], icon='🤖', accent=SAGE
            ),
        ], style={'padding':'0 16px'}),
        html.Div([
            html.Div([dcc.Graph(figure=fig_models)], style=HALF),
            html.Div([dcc.Graph(figure=fig_imp)],    style=HALF),
        ], style=ROW),
        html.Div([
            html.Div([dcc.Graph(figure=fig_avp_lr)], style={'flex':'1','minWidth':'300px'}),
            html.Div([dcc.Graph(figure=fig_avp_rf)], style={'flex':'1','minWidth':'300px'}),
            html.Div([dcc.Graph(figure=fig_avp_gb)], style={'flex':'1','minWidth':'300px'}),
        ], style=ROW),
        html.Div([
            story_panel(
                'How to Read the Feature Importance Chart', [
                    html.P([
                        html.B('Population accounts for 93.8% (Random Forest) and 97.8% (Gradient Boosting) '
                               'of all predictive importance. '),
                        'This is consistent with correlation analysis (r = 0.937), PCA (PC1 loading +0.559), '
                        'and regression (t = 81.7, p < .001). Every analytical method converges on the same '
                        'conclusion: population volume is the primary driver of trade area purchasing power.'
                    ]),
                    html.P([
                        html.B('Income contributes less than 1% of feature importance '),
                        'across both models — consistent with its near-zero PC1 loading (−0.027). '
                        'This directly challenges its role as a co-equal site-selection criterion in the '
                        'existing DNA model. Income is better used as a post-selection tenant-strategy input, '
                        'not a primary investment filter.'
                    ]),
                ], icon='⚖️', accent=SAGE
            ),
        ], style={'padding':'0 16px'}),
    ], style=SECTION_STYLE),

    # ════════════════════════════════════════════════════
    # SECTION 5 — TOP MARKETS & STRATEGIC PRIORITIES
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('5', 'Strategic Priorities — Top Trade Areas by PPI', AMBER),
        html.Div([
            insight_box(
                'Actionable Output: Use the Division and Income filters above to instantly generate a '
                'ranked priority list for any market segment. This chart updates dynamically — applying '
                'any strategic lens without code changes.',
                AMBER
            ),
        ], style={'padding':'0 16px'}),
        dcc.Graph(id='bar-top20', style={'padding':'0 8px'}),
    ], style=SECTION_STYLE),

    # ════════════════════════════════════════════════════
    # PAGE 6 — KEY CONCLUSIONS
    # ════════════════════════════════════════════════════
    html.Div([
        section_header('6', 'Key Conclusions & Strategic Recommendations', NAVY),
        html.Div([

            html.Div([
                story_panel(
                    'Conclusion 1 — Market Size is the Primary Driver', [
                        html.P(
                            'Population volume is the overwhelmingly dominant determinant of trade area '
                            'purchasing power, confirmed by four independent analytical methods: '
                            'correlation analysis (r = 0.937), PCA (PC1 loading +0.559), regression '
                            '(t = 81.7, p < .001), and ML feature importance (93.8%–97.8%). Median Income '
                            'contributes less than 1% of predictive importance.'
                        ),
                        html.P([
                            html.B('Recommendation: '),
                            'Establish population density as the mandatory first-stage site-selection filter. '
                            'Tier 1: above ~140,000 (75th percentile). Tier 4: below ~60,000 (25th percentile).'
                        ]),
                    ], icon='🏙️', accent=NAVY
                ),
                story_panel(
                    'Conclusion 2 — Daytime Population Matters Equally', [
                        html.P(
                            'Daytime population has a PC1 loading of +0.559 — equal to residential population. '
                            'For grocery-anchored retail, commuter and workforce traffic is often the primary '
                            'revenue driver, especially near employment centers, transit hubs, and medical campuses.'
                        ),
                        html.P([
                            html.B('Recommendation: '),
                            'Add daytime population to the site assessment scorecard with equal weight to '
                            'residential population. This requires no new data infrastructure.'
                        ]),
                    ], icon='🚇', accent=STEEL
                ),
                story_panel(
                    'Conclusion 3 — Income Belongs in Tenant Strategy', [
                        html.P(
                            'Median Income (r = 0.108; PC1 loading −0.027; <1% ML importance) adds noise '
                            'rather than signal to site selection. It operates on an independent dimension (PC2) '
                            'that measures spending quality, not spending volume.'
                        ),
                        html.P([
                            html.B('Recommendation: '),
                            'Remove income from site-selection scoring. Use it exclusively as a post-selection '
                            'tenant-strategy input: high income → premium/specialty retail; high population + '
                            'moderate income → grocery anchors and value convenience.'
                        ]),
                    ], icon='💰', accent=TERRA
                ),
            ], style={'flex':'1','minWidth':'360px','padding':'0 8px'}),

            html.Div([
                story_panel(
                    'Conclusion 4 — Deploy the Recommended Models', [
                        html.Div([
                            html.Div([
                                html.B('Gradient Boosting', style={'color':SAGE}),
                                html.P('For operational scoring of new trade areas', style={'fontSize':'12px','margin':'2px 0 0 0'}),
                                html.P('R² = 0.936  |  MAE = 0.163 PPI units', style={'fontSize':'12px','color':NAVY,'fontWeight':'bold','margin':'2px 0 0 0'}),
                            ], style={'flex':'1','padding':'10px','background':'#EBF7EF','borderRadius':'6px','margin':'4px 4px 0 0'}),
                            html.Div([
                                html.B('Linear Regression', style={'color':STEEL}),
                                html.P('For stakeholder communication & investment rationale', style={'fontSize':'12px','margin':'2px 0 0 0'}),
                                html.P('CV R² = 0.947  |  Interpretable coefficients', style={'fontSize':'12px','color':NAVY,'fontWeight':'bold','margin':'2px 0 0 0'}),
                            ], style={'flex':'1','padding':'10px','background':'#EBF3FB','borderRadius':'6px','margin':'4px 0 0 4px'}),
                        ], style={'display':'flex','marginBottom':'8px'}),
                        html.P(
                            'The dual-model approach correctly separates predictive accuracy (Gradient '
                            'Boosting) from interpretability (Linear Regression) — both required for '
                            'practical deployment in a commercial real estate investment context.',
                            style={'fontSize':'12px','margin':'4px 0 0 0'}
                        ),
                    ], icon='🎯', accent=SAGE
                ),
                story_panel(
                    'Conclusion 5 — Build an Emerging Market Pipeline', [
                        html.P(
                            'Markets with above-average income growth AND above-average household growth '
                            'simultaneously — concentrated in Central and West divisions — represent the '
                            'highest-priority forward-looking opportunities. Their current PPI scores may '
                            'be modest, but dual-growth dynamics position them for higher purchasing capacity '
                            'within 3–5 years.'
                        ),
                        html.P([
                            html.B('Recommendation: '),
                            'Establish a formal quarterly review process tracking PPI trajectory, population '
                            'growth realisation, and occupancy performance for these emerging candidates.'
                        ]),
                    ], icon='📈', accent=AMBER
                ),
                story_panel(
                    'Limitations & Future Work', [
                        html.Ul([
                            html.Li([html.B('Circular definition: '), 'Income × Population shares variables with model predictors — log-transformation of PPI recommended']),
                            html.Li([html.B('NYC metro outliers: '), 'Two properties (PPI = 18.01, 17.76) cause normality violations and dominate portfolio metrics']),
                            html.Li([html.B('Cross-sectional data: '), 'Panel modeling using 2017–2024 historical data would detect PPI trajectories over time']),
                            html.Li([html.B('Missing context: '), 'Competitive retail density, foot traffic data, and physical accessibility data would enable capturable demand estimation']),
                        ], style={'paddingLeft':'18px','fontSize':'12px','margin':'0'})
                    ], icon='⚠️', accent=GREY
                ),
            ], style={'flex':'1','minWidth':'360px','padding':'0 8px'}),

        ], style={**ROW,'padding':'0 8px'}),
    ], style=SECTION_STYLE),

    # Footer
    html.Div([
        html.P(
            'Regency Centers Trade Area Purchasing Power Dashboard  |  '
            'CEN6940.14925 Data Analytics and Decision Making  |  '
            'Group 7 — University of North Florida  |  Spring 2026',
            style={'color':GREY,'fontSize':'11px','margin':'0','fontFamily':'Arial','textAlign':'center'}
        )
    ], style={'padding':'16px','borderTop':f'1px solid #dde3ee','marginTop':'8px'}),

], style={
    'fontFamily': 'Arial, sans-serif',
    'backgroundColor': '#EDF2FA',
    'maxWidth': '1440px',
    'margin': '0 auto',
    'padding': '0 0 20px 0'
})


# ════════════════════════════════════════════════════════
# CALLBACK
# ════════════════════════════════════════════════════════
@app.callback(
    [
        Output('kpi-count',       'children'),
        Output('kpi-income',      'children'),
        Output('kpi-pop',         'children'),
        Output('kpi-ppi',         'children'),
        Output('kpi-pp',          'children'),
        Output('kpi-incgrow',     'children'),
        Output('kpi-hhgrow',      'children'),
        Output('bar-top-markets', 'figure'),
        Output('bar-div-pp',      'figure'),
        Output('bar-div-ppi',     'figure'),
        Output('box-div-ppi',     'figure'),
        Output('scatter-pop',     'figure'),
        Output('scatter-income',  'figure'),
        Output('scatter-daytime', 'figure'),
        Output('scatter-growth',  'figure'),
        Output('scatter-ppi',     'figure'),
        Output('hist-ppi',        'figure'),
        Output('bar-top20',       'figure'),
    ],
    [Input('div-filter', 'value'), Input('income-slider', 'value')]
)
def update_all(div_val, income_range):
    f = data.copy()
    if div_val != 'ALL':
        f = f[f['Division'] == div_val]
    f = f[(f['Income'] >= income_range[0]) & (f['Income'] <= income_range[1])].reset_index(drop=True)
    n = len(f)

    if n == 0:
        empty = go.Figure()
        empty.add_annotation(text='No data for selected filters', x=0.5, y=0.5, showarrow=False, font_size=14)
        return ('N/A',)*7 + (empty,)*11

    kpi_count   = f'{n:,}'
    kpi_income  = f"${f['Income'].mean():,.0f}"
    kpi_pop     = f"{f['Population'].mean():,.0f}"
    kpi_ppi     = f"{f['PPI'].mean():.2f}"
    kpi_pp      = f"${f['PP_Billions'].sum():,.1f}B"
    kpi_incgrow = f"{f['Income_Growth'].mean()*100:.1f}%"
    kpi_hhgrow  = f"{f['HH_Growth'].mean()*100:.1f}%"

    # ── Section 1 ────────────────────────────────────────
    top10  = f.nlargest(10, 'PP_Billions').reset_index(drop=True)
    names10 = [str(nm)[:30]+'\u2026' if len(str(nm))>30 else str(nm) for nm in top10['Name']]
    fig1a = go.Figure(go.Bar(
        x=top10['PP_Billions'], y=names10, orientation='h',
        marker_color=[DIV_COLORS.get(d,GREY) for d in top10['Division']],
        text=[f'${v:.1f}B' for v in top10['PP_Billions']], textposition='outside',
        customdata=top10['Division'],
        hovertemplate='%{y}<br>PP: %{x:.1f}B<br>Division: %{customdata}<extra></extra>'
    ))
    fig1a.update_layout(title='Figure 1: Top 10 Markets by Purchasing Power ($B)',
        xaxis_title='Purchasing Power ($B)', font_family='Georgia', title_font_size=13,
        margin=dict(t=45,b=10,l=10,r=60), yaxis=dict(autorange='reversed'))

    div_pp = f.groupby('Division').agg(Avg_PP=('PP_Billions','mean')).reset_index().sort_values('Avg_PP',ascending=False)
    fig1b = go.Figure(go.Bar(
        x=div_pp['Division'], y=div_pp['Avg_PP'],
        marker_color=[DIV_COLORS.get(d,GREY) for d in div_pp['Division']],
        text=[f'${v:.1f}B' for v in div_pp['Avg_PP']], textposition='outside'
    ))
    fig1b.update_layout(title='Figure 2: Average Purchasing Power by Division',
        yaxis_title='Avg PP ($B)', font_family='Georgia', title_font_size=13,
        margin=dict(t=45,b=10,l=10,r=10))

    div_ppi = f.groupby('Division').agg(Avg_PPI=('PPI','mean')).reset_index().sort_values('Avg_PPI',ascending=False)
    fig1c = go.Figure(go.Bar(
        x=div_ppi['Division'], y=div_ppi['Avg_PPI'],
        marker_color=[DIV_COLORS.get(d,GREY) for d in div_ppi['Division']],
        text=[f'{v:.2f}' for v in div_ppi['Avg_PPI']], textposition='outside'
    ))
    fig1c.update_layout(title='Figure 3: Average PPI Score by Division',
        yaxis_title='Average PPI', font_family='Georgia', title_font_size=13,
        margin=dict(t=45,b=10,l=10,r=10))

    box_traces = []
    for div in sorted(f['Division'].unique()):
        vals = f[f['Division']==div]['PPI'].values
        box_traces.append(go.Box(y=vals, name=div, marker_color=DIV_COLORS.get(div,GREY), line_width=1.5))
    fig1d = go.Figure(data=box_traces)
    fig1d.update_layout(title='Figure 4: PPI Distribution by Division',
        yaxis_title='PPI Score (Standardised)', showlegend=False,
        font_family='Georgia', title_font_size=13, margin=dict(t=45,b=10,l=10,r=10))

    # ── Section 2 ────────────────────────────────────────
    def scatter_fig(x_col, y_col, title, x_lbl, y_lbl='Purchasing Power ($B)'):
        fig = go.Figure()
        for div in sorted(f['Division'].unique()):
            df_d = f[f['Division']==div]
            fig.add_trace(go.Scatter(
                x=df_d[x_col], y=df_d[y_col], mode='markers', name=div,
                marker=dict(color=DIV_COLORS.get(div,GREY), size=5, opacity=0.65),
                hovertemplate=f'%{{text}}<br>{x_lbl}: %{{x:,.0f}}<br>{y_lbl}: %{{y:.2f}}<extra></extra>',
                text=df_d['Name']
            ))
        if len(f)>2:
            xv = f[x_col].values; yv = f[y_col].values
            mask = np.isfinite(xv) & np.isfinite(yv)
            if mask.sum()>2:
                m, b = np.polyfit(xv[mask], yv[mask], 1)
                xl = np.array([xv[mask].min(), xv[mask].max()])
                fig.add_trace(go.Scatter(x=xl, y=m*xl+b, mode='lines',
                    line=dict(color='#555555',dash='dash',width=1.5), name='Trend', showlegend=False))
        fig.update_layout(title=title, xaxis_title=x_lbl, yaxis_title=y_lbl,
            legend=dict(x=0.01,y=0.99,bgcolor='rgba(255,255,255,0.8)'),
            font_family='Georgia', title_font_size=13, margin=dict(t=45,b=10,l=10,r=10))
        return fig

    fig2a = scatter_fig('Population',   'PP_Billions', 'Figure 1a: Population vs. Purchasing Power',   'Population (2025)')
    fig2b = scatter_fig('Income',       'PP_Billions', 'Figure 1b: Median Income vs. Purchasing Power','Median HH Income ($)')
    fig2c = scatter_fig('Daytime_Pop',  'PP_Billions', 'Figure 1c: Daytime Population vs. Purchasing Power','Daytime Population (2025)')

    fig2d = go.Figure()
    for div in sorted(f['Division'].unique()):
        df_d = f[f['Division']==div]
        fig2d.add_trace(go.Scatter(
            x=df_d['Income_Growth']*100, y=df_d['HH_Growth']*100, mode='markers', name=div,
            marker=dict(color=DIV_COLORS.get(div,GREY),
                        size=df_d['PP_Billions'].clip(upper=30)*1.2+4, opacity=0.6, line=dict(width=0)),
            hovertemplate='%{text}<br>Income Growth: %{x:.1f}%<br>HH Growth: %{y:.1f}%<extra></extra>',
            text=df_d['Name']
        ))
    fig2d.update_layout(title='Figure 1d: Income Growth vs. HH Growth (bubble = Purchasing Power)',
        xaxis_title='Income Growth Rate (%)', yaxis_title='Household Growth Rate (%)',
        legend=dict(x=0.01,y=0.99,bgcolor='rgba(255,255,255,0.8)'),
        font_family='Georgia', title_font_size=13, margin=dict(t=45,b=10,l=10,r=10))

    # ── Section 3 ────────────────────────────────────────
    fig3a = scatter_fig('Population', 'PPI',
        'Figure 2a: Population vs. Purchasing Power Index (PPI)',
        'Population (2025)', 'PPI Score (PC1)')

    fig3b = go.Figure()
    for div in sorted(f['Division'].unique()):
        vals = f[f['Division']==div]['PPI'].values
        fig3b.add_trace(go.Histogram(x=vals, name=div,
            marker_color=DIV_COLORS.get(div,GREY), opacity=0.65, nbinsx=30))
    fig3b.update_layout(title='Figure 2b: PPI Score Distribution by Division',
        xaxis_title='PPI Score', yaxis_title='Count', barmode='overlay',
        legend=dict(x=0.75,y=0.99,bgcolor='rgba(255,255,255,0.8)'),
        font_family='Georgia', title_font_size=13, margin=dict(t=45,b=10,l=10,r=10))

    # ── Section 5 ────────────────────────────────────────
    top20  = f.nlargest(20, 'PPI').reset_index(drop=True)
    names20 = [str(nm)[:35]+'\u2026' if len(str(nm))>35 else str(nm) for nm in top20['Name']]
    fig5 = go.Figure(go.Bar(
        x=names20, y=top20['PPI'],
        marker_color=[DIV_COLORS.get(d,GREY) for d in top20['Division']],
        text=[f'{v:.2f}' for v in top20['PPI']], textposition='outside',
        customdata=top20['Division'],
        hovertemplate='%{x}<br>PPI: %{y:.2f}<br>Division: %{customdata}<extra></extra>'
    ))
    fig5.update_layout(title='Figure 10: Top 20 Trade Areas by Purchasing Power Index (PPI)',
        yaxis_title='PPI Score', xaxis_tickangle=-35,
        font_family='Georgia', title_font_size=13, margin=dict(t=50,b=130,l=10,r=10))

    return (kpi_count, kpi_income, kpi_pop, kpi_ppi, kpi_pp, kpi_incgrow, kpi_hhgrow,
            fig1a, fig1b, fig1c, fig1d, fig2a, fig2b, fig2c, fig2d, fig3a, fig3b, fig5)


# ════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════
app.run(mode='inline', height=1600)
